# Virtual Philadelphia

### basic imports

In [1]:
import subprocess
import sys
import shutil
import os
from datetime import datetime

### install local dependencies and build pol.jar file

In [2]:
CURRENT_DIR = os.path.dirname(os.path.abspath('large-scale-dataset-generator-for-nomad'))
PARENT_DIR = os.path.dirname(CURRENT_DIR)

process = subprocess.Popen(
    ["bash", "mvn.sh", "full"],
    cwd=PARENT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

for line in process.stdout:
    print(line, end="")

installing local dependencies
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

# All requested packages already installed.

Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
63 packages excluded due to repository priority protections
No packages marked for update
Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
63 packages excluded due to repository priority protections
Package maven-3.0.5-17.amzn2.noarch already installed and latest version
Nothing to do
[INFO] Scanning for projects...
[INFO]                                                                         
[INFO] ------------------------------------------------------------------------
[INFO] Building pol 0.2
[INFO] ------------------------------------------------------------------------
[INFO] 
[INFO] --- maven-install-plugin:2.3

### install utility modules

In [3]:
import map_generation.utils.pqgis as pqgis
import utils.file as file
from s3_upload import upload_to_s3
from convert_to_parquet import convert_travel_journal_to_parquet, convert_trajectories_to_parquet
from sparsify_parquet import sparsify_trajectories_parquet
from generate_poi_table import generate_poi_table

### generate run ID and create directory structure

In [4]:
RUN_DIR = "geolife_plus_phl"
LOGS_DIR = os.path.join(RUN_DIR, "logs")
PARQUET_DIR = os.path.join(RUN_DIR, "parquet")

if os.path.exists(RUN_DIR):
    shutil.rmtree(RUN_DIR)

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)

# print("RUN_ID:", RUN_ID)
print("RUN_DIR:", RUN_DIR)
print("LOGS_DIR:", LOGS_DIR)
print("PARQUET_DIR:", PARQUET_DIR)

RUN_DIR: geolife_plus_phl
LOGS_DIR: geolife_plus_phl/logs
PARQUET_DIR: geolife_plus_phl/parquet


### Set S3 config here (enter AWS profile name where it says s3_profile)

In [5]:
s3_bucket = "catalog-onspatial"
s3_traj_prefix = f"{RUN_DIR}/device-level/trajectories"
s3_sparse_prefix = f"{RUN_DIR}/device-level/trajectories_sparse"
s3_diaries_prefix = f"{RUN_DIR}/diaries"
s3_poi_prefix = f"{RUN_DIR}/poi_table"
s3_profile = "707813031043-PennResearchAssistant"

### set bounding box and output folder for generated map

In [6]:
bounding_box = [-75.19747721789525, 39.931392279878246, -75.14652246706544, 39.96336810441389]
output_folder = '/run/user/1000/maps/philadelphia'

In [8]:
pqgis.generate_map(bounding_box, output_folder, new_map=True)

Getting buildings geojson from overpass
Getting walkways geojson from overpass
Generating map shapefiles
Getting buildings shapefile
  Business abandoned abandoned:power access     addr:city addr:country  \
0     None      None            None   None  Philadelphia         None   
1     None      None            None   None  Philadelphia         None   
2     None      None            None   None          None         None   
3     None      None            None   None  Philadelphia         None   
4     None      None            None   None  Philadelphia         None   

     addr:housename addr:housenumber addr:housenumber_1 addr:housenumber_2  \
0              None              520               None               None   
1              None              340               None               None   
2  Edwards Building             None               None               None   
3              None             3701               None               None   
4              None             

In [9]:
# copy map to local directory
shutil.copytree('/run/user/1000/maps/philadelphia', 'maps/philadelphia', dirs_exist_ok=True)

'maps/philadelphia'

### edit modified.properties to set any relevant parameters before continuing

In [10]:
def merge_properties(base_path, override_path, output_path):
    merged = {}

    def load_file(path):
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if "=" not in line or line.startswith("#"):
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                merged[key] = f"{key}={value.strip()}"

    # Load files in order – second file overrides first
    load_file(base_path)
    load_file(override_path)

    # Write the merged output
    with open(output_path, "w") as f:
        for line in merged.values():
            f.write(line + "\n")


# Usage
MERGED_CONFIG = "./merged.properties"
merge_properties("../parameters.properties", "modified.properties", MERGED_CONFIG)

### launch simulation

In [11]:
# SET LENGTH OF SIMULATION = simulation_time * time step (default = 5 min, multiply 288 by number of days)
simulation_time = "2880"

cmd = [
    "java",
    "-Dpol.gui=false",
    "-Djava.awt.headless=true",
    "-Dlog4j2.configurationFactory=pol.log.CustomConfigurationFactory",
    f"-Dlog.rootDirectory={RUN_DIR}",
    f"-Dsimulation.test=all",
    "-jar", "../jar/pol.jar",
    "-configuration", MERGED_CONFIG,
    "-until", simulation_time
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

# Stream output line by line as it happens
for line in process.stdout:
    print(line, end="")

# Wait for process to complete and get return code
return_code = process.wait()

print("-" * 80)
if return_code == 0:
    print("✓ Simulation completed successfully")
else:
    print(f"✗ Simulation failed with return code: {return_code}")

MASON Version 19.  For further options, try adding ' -help' at end.
Dec 16, 2025 11:02:26 PM org.apache.commons.beanutils.FluentPropertyBeanIntrospector introspect
INFO: Error when creating PropertyDescriptor for public final void org.apache.commons.configuration2.AbstractConfiguration.setProperty(java.lang.String,java.lang.Object)! Ignoring this property.
[INIT] WorldModel construction starting with 200 agents
[INIT] Loading map layers...
Entry: stylesheet/social.css
Copying Resource Name.../stylesheet/social.css
Entry: stylesheet/NodeColoringBasedOnInterest.css
Copying Resource Name.../stylesheet/NodeColoringBasedOnInterest.css
file:/home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/maps/philadelphia/map/walkways.shp
file:/home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/maps/philadelphia/map/buildings.shp
file:/home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/maps/philadelphia/map/buildingUnits.shp
[INIT] Map lay

### generate trajectories using integrate.py

In [12]:
file_prefix = "AgentStateTable"
appended_file = "trajectories.tsv"
print(f"Integrating log files from {LOGS_DIR} with prefix {file_prefix} into {appended_file}")
if file_prefix=='Checkin':
    file.integrate_log_files(f"/home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/{LOGS_DIR}", 
                             file_prefix, f"/home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/{LOGS_DIR}/trajectories.tsv", remove_original=True)
elif file_prefix=='AgentStateTable':
    file.integrate_log_files(f"/home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/{LOGS_DIR}", 
                             file_prefix, f"/home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/{LOGS_DIR}/trajectories.tsv", command='cut -f 1-4')

Integrating log files from geolife_plus_phl/logs with prefix AgentStateTable into trajectories.tsv
Appended /home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/geolife_plus_phl/logs/AgentStateTable.tsv to /home/ec2-user/SageMaker/large-scale-dataset-generator-for-nomad/headless/geolife_plus_phl/logs/trajectories.tsv
0    2025-07-01T00:00:00.000
1    2025-07-01T00:00:00.000
2    2025-07-01T00:00:00.000
3    2025-07-01T00:00:00.000
4    2025-07-01T00:00:00.000
Name: simulationTime, dtype: object
Column 1 is sorted: True


### convert trajectories and stops to parquet, upload to S3 (set up AWS config beforehand)

In [13]:
travel_journal_csv = os.path.join(LOGS_DIR, "TravelJournal.csv")
trajectories_tsv = os.path.join(LOGS_DIR, "trajectories.tsv")
base_output_dir = PARQUET_DIR

try:
    # Convert TravelJournal.csv to parquet
    print("=" * 60)
    print("CONVERTING TRAVEL JOURNAL")
    print("=" * 60)
    travel_journal_dir = convert_travel_journal_to_parquet(travel_journal_csv, f"{base_output_dir}/travel_journal")
        
    # Convert trajectories.tsv to parquet
    print("\n" + "=" * 60)
    print("CONVERTING TRAJECTORIES")
    print("=" * 60)
    trajectories_dir = convert_trajectories_to_parquet(trajectories_tsv, f"{base_output_dir}/trajectories")
        
    # Upload to S3 if bucket specified
    if s3_bucket:
        print("\n" + "=" * 60)
        print("UPLOADING TO S3")
        print("=" * 60)
            
        # Upload travel journal
        print("Uploading Travel Journal...")
        upload_to_s3(travel_journal_dir, s3_bucket, s3_diaries_prefix, s3_profile)
            
        # Upload trajectories
        print("\nUploading Trajectories...")
        upload_to_s3(trajectories_dir, s3_bucket, s3_traj_prefix, s3_profile)
    else:
        print("\nSkipping S3 upload (no bucket specified)")
            
except Exception as e:
    print(f"Error during conversion: {e}")

CONVERTING TRAVEL JOURNAL
Reading TravelJournal.csv from: geolife_plus_phl/logs/TravelJournal.csv
Loaded 14928 records
Parsing geometry coordinates...
Processing 14928 records with valid coordinates
Writing partitioned parquet files...
  Wrote 2699 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-01/part-0.parquet
  Wrote 1110 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-02/part-0.parquet
  Wrote 1237 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-03/part-0.parquet
  Wrote 1496 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-04/part-0.parquet
  Wrote 1578 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-05/part-0.parquet
  Wrote 1520 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-06/part-0.parquet
  Wrote 1461 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-07/part-0.parquet
  Wrote 1374 records to geolife_plus_phl/parquet/travel_journal/date=2025-07-08/part-0.parquet
  Wr

### sparsify the trajectories using sparsify_parquet.py and upload to S3

In [14]:
# Sparsify trajectory parquet files using thin_traj_by_times.

# Reads partitioned parquet files created by convert_to_parquet.py and applies
# trajectory sparsification using ping times generated with realistic burst/gap patterns
# or uniform sampling.

# Usage:
#   python sparsify_parquet.py <input_parquet_dir> <output_parquet_dir> [options]

# Options:
#   --beta-start MINUTES      Mean time between bursts (default: 120 = 2 hours)
#   --beta-durations MINUTES  Mean burst duration (default: 30 minutes)
#   --beta-ping MINUTES       Mean time between pings within burst (default: 5 minutes)
#   --uniform MINUTES         Use uniform sampling every N minutes (ignores burst params)
#   --seed INT                Random seed for reproducibility (default: 42)
#   --no-deduplicate          Don't remove duplicate trajectory indices

# Example:
#   # Realistic mobile phone pattern (bursts every ~2 hours, lasting ~30 min, pings every ~5 min)
#   python sparsify_parquet.py data/parquet/trajectories data/parquet/trajectories_sparse
  
#   # Uniform sampling every 15 minutes
#   python sparsify_parquet.py data/parquet/trajectories data/parquet/trajectories_sparse --uniform 15
  
#   # Custom burst pattern
#   python sparsify_parquet.py data/parquet/trajectories data/parquet/trajectories_sparse --beta-start 60 --beta-durations 20 --beta-ping 3

In [15]:
# set options
beta_start = 300
beta_durations = 55
beta_ping = 7
seed = 0
uniform = None
deduplicate = True
output_bursts = False

try:
    sparsify_trajectories_parquet(
        input_dir=f"{PARQUET_DIR}/trajectories",
        output_dir=f"{PARQUET_DIR}/trajectories_sparse",
        beta_start=beta_start,
        beta_durations=beta_durations,
        beta_ping=beta_ping,
        uniform_minutes=uniform,
        seed=seed,
        deduplicate=deduplicate,
        output_bursts=output_bursts
    )
        
    # Upload to S3 if bucket specified
    try:
        upload_to_s3(f"{PARQUET_DIR}/trajectories_sparse", s3_bucket, s3_sparse_prefix, s3_profile)
    except Exception as e:
        print("S3 upload failed")
            
except Exception as e:
    print(f"Error during sparsification: {e}")
    import traceback
    traceback.print_exc()

Sparsification parameters:
  Mode: Burst pattern
    beta_start: 300 minutes (time between bursts)
    beta_durations: 55 minutes (burst duration)
    beta_ping: 7 minutes (time between pings)
  seed: 0 (base seed, per-user seeds will be sequential)
  deduplicate: True

Reading all parquet files from geolife_plus_phl/parquet/trajectories...
Loaded 576400 records from all partitions
Processing 200 unique users...
  Processed 200/200 users.
Combining all sparsified trajectories...
Writing output parquet files...
  Wrote partitioned parquet to geolife_plus_phl/parquet/trajectories_sparse

SPARSIFICATION COMPLETE
Total input records: 576400
Total output records: 46645
Overall reduction: 91.9%
Output written to: geolife_plus_phl/parquet/trajectories_sparse

Uploading to S3 bucket: catalog-onspatial
S3 prefix: geolife_plus_phl/device-level/trajectories_sparse
AWS profile: 707813031043-PennResearchAssistant
S3 upload failed


### generate POI table (visited by agents) and upload to S3

In [16]:
try:
    generate_poi_table(LOGS_DIR, f"{RUN_DIR}/poi_table_visited.parquet")
    try:
        upload_to_s3(f"{RUN_DIR}/poi_table_visited.parquet", s3_bucket, s3_poi_prefix, s3_profile)
    except Exception as e:
        print("S3 upload failed")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

GENERATING POI TABLE
Loading ApartmentTable.tsv...
  Loaded 300 apartments
Loading WorkplaceTable.tsv...
  Loaded 50 workplaces
Loading RestaurantTable.tsv...
  Loaded 4 restaurants
Loading PubTable.tsv...
  Loaded 2 pubs
Loading ClassroomTable.tsv...
  Loaded 1 classrooms

Total BuildingUnits loaded: 357

Loading TravelJournal.csv...
Found 241 unique visited locations

Filtering to visited locations...
POIs after filtering: 241

Parsing geometry coordinates...
POIs with valid coordinates: 241

Writing POI table to parquet: geolife_plus_phl/poi_table_visited.parquet

POI TABLE GENERATION COMPLETE
Total POIs: 241
Output file: geolife_plus_phl/poi_table_visited.parquet

POIs by venue type:
venue_type
apartment     186
workplace      49
restaurant      4
pub             2

Sample POIs (first 5):
   id venue_type              x             y                                      geometry  buildingId  neighborhoodId  attractiveness  personCapacity
0   1  apartment  486433.878139  4.421114e+0